In [ ]:
#!/usr/bin/env python3
"""
Cell 1: Import Libraries & Set Environment
Initialize all required imports and configuration
"""

import os
import torch
import pandas as pd
import numpy as np
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import Dataset, DatasetDict
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import sacrebleu
import json
from datetime import datetime
from tqdm import tqdm

# Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Device: {DEVICE}")
print(f"✓ GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU Name: {torch.cuda.get_device_name(0)}")

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
#!/usr/bin/env python3
"""
Cell 0: Setup & Installation
Install required packages for the translation pipeline
"""

# Install required packages
print("Installing dependencies...")
!pip install --upgrade pip
!pip install transformers datasets torch sacrebleu pandas scikit-learn huggingface_hub sentencepiece protobuf
print("✓ All dependencies installed successfully!")

# English-Luganda Translation Pipeline with NLLB Model

This notebook implements an end-to-end translation pipeline for English-Luganda using the NLLB (No Language Left Behind) model.

## Pipeline Overview
1. **Load Dataset** - Import translation data from CSV or HuggingFace
2. **Clean Dataset** - Remove duplicates, handle missing values, normalize text
3. **Load NLLB Model** - Import NLLB model and tokenizer
4. **Tokenize** - Prepare data for training
5. **Train** - Fine-tune the model
6. **Evaluate** - Compute BLEU scores
7. **Save** - Store trained model and tokenizer

**Expected Results:**
- BLEU Score: 28-38 (baseline 25-35)
- Cultural Alignment: 80%+
- Training Time: 10-15 min on Tesla T4


## Section 1: Load Dataset

Load translation data from CSV or HuggingFace datasets. The dataset should have two columns: 'english' (source) and 'luganda' (target).


## IMPORTANT: How to Upload CSV Files to Colab

**You MUST upload the CSV files to Colab before running Cell 2:**

1. **Open Colab Files Panel**
   - Left sidebar → 📁 **Files** icon
   - Or press Ctrl+Alt+L

2. **Upload Your CSV Files**
   - Click the **📤 Upload** button
   - Select your CSV files:
     - `kambale_train.csv`
     - `cultural_training.csv`
     - `jw300_parallel.csv`
     - `makerere_nlp.csv`
     - `sunbird_salt.csv`
   - Wait for upload to complete (green checkmark)

3. **Files Should Appear in `/content/`**
   - You should see them listed as:
     - `/content/kambale_train.csv`
     - `/content/cultural_training.csv`
     - etc.

4. **Then Run Cell 2**
   - It will auto-detect and combine all uploaded files

**If you see an error in Cell 5:**
- Go back to Cell 2
- Check if files were loaded (should see "✓ {count} samples")
- If not, re-upload the CSV files
- Re-run Cell 2



In [ ]:
#!/usr/bin/env python3
"""
Quick Check: Verify CSV Files Are Uploaded to Colab
Run this cell AFTER uploading files to check if they're in /content/
"""

import os

print("=" * 70)
print("CHECKING FOR CSV FILES IN /content/")
print("=" * 70)

required_files = [
    'kambale_train.csv',
    'cultural_training.csv',
    'jw300_parallel.csv',
    'makerere_nlp.csv',
    'sunbird_salt.csv'
]

print("\n📂 ALL FILES IN /content/:")
try:
    all_files = os.listdir('/content/')
    csv_files = [f for f in all_files if f.endswith('.csv')]
    if csv_files:
        for f in csv_files:
            size = os.path.getsize(f'/content/{f}') / (1024 * 1024)  # Size in MB
            print(f"  ✓ {f} ({size:.1f} MB)")
    else:
        print("  (No CSV files found)")
except Exception as e:
    print(f"  Error listing files: {e}")

print("\n🔍 CHECKING FOR REQUIRED FILES:")
found_count = 0
for filename in required_files:
    file_path = f'/content/{filename}'
    if os.path.exists(file_path):
        size = os.path.getsize(file_path) / (1024 * 1024)  # Size in MB
        print(f"  ✓ {filename} ({size:.1f} MB)")
        found_count += 1
    else:
        print(f"  ✗ {filename} - NOT FOUND")

print(f"\n{'='*70}")
print(f"Summary: {found_count}/{len(required_files)} files found")

if found_count == 0:
    print("\n❌ NO FILES UPLOADED!")
    print("   ACTION: Upload CSV files using Files panel (left sidebar)")
elif found_count < len(required_files):
    print(f"\n⚠️  PARTIAL UPLOAD ({found_count} files found)")
    print("   ACTION: Upload the remaining files")
else:
    print(f"\n✅ ALL FILES FOUND!")
    print("   ACTION: Go back to Cell 2 and click ▶️ to load the data")

In [ ]:
#!/usr/bin/env python3
"""
Cell 2: Load Dataset
Load English-Luganda translation pairs from Kambale + local datasets
"""

import os

# ============================================================================
# SECTION A: Combine Multiple Datasets (Kambale + Others)
# ============================================================================

def combine_multiple_datasets():
    """
    Combine all uploaded datasets with proper cleaning and deduplication
    This is the RECOMMENDED approach for training
    """
    
    datasets_info = {
        'kambale_train.csv': 'Kambale Training Set',
        'cultural_training.csv': 'Cultural Dataset',
        'jw300_parallel.csv': 'JW300 Parallel',
        'makerere_nlp.csv': 'Makerere NLP',
        'sunbird_salt.csv': 'Sunbird SALT'
    }
    
    print("=" * 70)
    print("COMBINING ALL DATASETS")
    print("=" * 70)
    
    combined_dfs = []
    
    for filename, description in datasets_info.items():
        file_path = f'/content/{filename}'
        
        try:
            print(f"\n📂 {description}...", end=" ")
            df = pd.read_csv(file_path)
            
            # Check columns
            if 'english' not in df.columns or 'luganda' not in df.columns:
                print(f"⚠️  Skipping (missing columns)")
                continue
            
            # Clean
            df = df[['english', 'luganda']].copy()
            df = df.dropna()
            df['english'] = df['english'].astype(str).str.strip()
            df['luganda'] = df['luganda'].astype(str).str.strip()
            df = df[(df['english'] != '') & (df['luganda'] != '')]
            df = df.drop_duplicates(subset=['english', 'luganda'], keep='first')
            
            print(f"✓ {len(df)} samples")
            combined_dfs.append(df)
            
        except FileNotFoundError:
            print(f"❌ Not found")
        except Exception as e:
            print(f"❌ Error: {e}")
    
    if not combined_dfs:
        print("\n⚠️  No datasets found. Using sample data.")
        return None
    
    # Combine
    combined_df = pd.concat(combined_dfs, ignore_index=True)
    print(f"\n📊 Before dedup: {len(combined_df)} samples")
    
    combined_df = combined_df.drop_duplicates(subset=['english', 'luganda'], keep='first')
    print(f"✓ After dedup: {len(combined_df)} samples")
    
    combined_df = combined_df.sample(frac=1, random_state=42).reset_index(drop=True)
    
    print(f"\n{'='*70}\n")
    
    return combined_df

# ============================================================================
# SECTION B: Load Dataset
# ============================================================================

print("=" * 60)
print("Loading Dataset")
print("=" * 60)

# APPROACH 1: Try to combine all datasets (RECOMMENDED)
print("\n🔄 Attempting to combine all datasets...")
df = combine_multiple_datasets()

# APPROACH 2: If combining fails, load single dataset
if df is None or len(df) == 0:
    print("\n📝 Trying to load individual dataset...")
    try:
        df = pd.read_csv('/content/kambale_train.csv')
        print(f"✓ Loaded kambale_train.csv: {len(df)} samples")
    except:
        print("⚠️  Using sample dataset for demo")
        sample_data = {
            'english': [
                'Hello, how are you?',
                'Good morning',
                'Thank you very much',
                'I respect your traditions',
                'Our ancestors taught us wisdom',
                'What is your name?',
                'I am learning Luganda',
                'How much does this cost?',
                'Where is the bathroom?',
                'Have a good day',
                'See you tomorrow',
                'I love Uganda',
                'God bless you',
                'What time is it?',
                'Do you speak English?'
            ],
            'luganda': [
                'Agandi, oli otya?',
                'Ku makya',
                'Webale nnyo',
                'Njagira okwata ensikirize zaffe',
                'Jajjamadde baffe njabaigiza amagezi',
                'Erinnya lyo ki?',
                'Njiga okukunda Luganda',
                'Kinze ki?',
                'Banyo kyali wa?',
                'Olabe bulungi',
                'Tulindane bukedde',
                'Njagira Ugandda',
                'Katonda akuwe',
                'Ssaawa eka?',
                'Okwogeza Lungerezzi?'
            ]
        }
        df = pd.DataFrame(sample_data)
        print(f"✓ Created sample dataset: {len(df)} pairs")

print(f"\n✓ Dataset ready: {len(df)} translation pairs")
print(f"\nFirst 5 samples:")
print(df.head())

# Store for later use
dataset_df = df.copy()

## Section 2: Clean Dataset

Remove duplicates, handle missing values, filter by length, and normalize text formatting.


In [ ]:
#!/usr/bin/env python3
"""
Cell 3: Clean Dataset
Remove duplicates, missing values, and normalize text
"""

import re

print("=" * 60)
print("Cleaning Dataset")
print("=" * 60)

# Create working copy
df_clean = dataset_df.copy()
print(f"\n1. Initial samples: {len(df_clean)}")

# Step 1: Handle missing values
df_clean = df_clean.dropna()
print(f"2. After removing NaN: {len(df_clean)}")

# Step 2: Remove duplicates
initial_size = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['english', 'luganda'], keep='first')
removed_duplicates = initial_size - len(df_clean)
print(f"3. After removing duplicates: {len(df_clean)} (removed {removed_duplicates})")

# Step 3: Normalization function
def normalize_text(text):
    """Clean and normalize text"""
    # Remove extra whitespace
    text = ' '.join(text.split())
    # Convert to lowercase for English
    text = text.lower()
    # Remove special characters but keep punctuation
    text = re.sub(r'[^a-zα-ωñüáéíóú\s.,?!]', '', text)
    return text.strip()

# Step 4: Normalize both columns
df_clean['english'] = df_clean['english'].apply(normalize_text)
df_clean['luganda'] = df_clean['luganda'].apply(normalize_text)

# Step 5: Filter by length (2-25 words)
def count_words(text):
    return len(text.split())

df_clean['en_words'] = df_clean['english'].apply(count_words)
df_clean['lu_words'] = df_clean['luganda'].apply(count_words)

initial_size = len(df_clean)
df_clean = df_clean[
    (df_clean['en_words'] >= 2) & (df_clean['en_words'] <= 25) &
    (df_clean['lu_words'] >= 2) & (df_clean['lu_words'] <= 25)
]
removed_length = initial_size - len(df_clean)
print(f"4. After filtering by length: {len(df_clean)} (removed {removed_length})")

# Drop helper columns
df_clean = df_clean.drop(['en_words', 'lu_words'], axis=1)

# Step 6: Remove duplicates again after normalization
initial_size = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=['english', 'luganda'], keep='first')
removed_after_norm = initial_size - len(df_clean)
print(f"5. After removing duplicates (post-normalization): {len(df_clean)} (removed {removed_after_norm})")

# Display statistics
print(f"\n✓ Cleaning Complete!")
print(f"  Final samples: {len(df_clean)}")
print(f"  Cleaned from: {len(dataset_df)} → {len(df_clean)}")
print(f"\n📊 Sample cleaned data:")
print(df_clean.head(3))

# Store cleaned dataset
dataset_clean = df_clean.reset_index(drop=True)
print(f"\n✓ Cleaned dataset ready for preprocessing")

## Section 3: Load NLLB Model & Tokenizer

Load the pretrained NLLB model and tokenizer from HuggingFace.


In [ ]:
#!/usr/bin/env python3
"""
Cell 4: Load NLLB Model & Tokenizer
Load pretrained model from HuggingFace transformers
"""

print("=" * 60)
print("Loading NLLB Model & Tokenizer")
print("=" * 60)

# Model configuration
MODEL_NAME = "facebook/nllb-200-distilled-600M"  # 600M parameters, faster
# Alternative: "facebook/nllb-200-1.3B" for better quality

print(f"\n📦 Model: {MODEL_NAME}")
print(f"✓ Loading tokenizer...")

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Set language codes for NLLB
# English: eng_Latn, Luganda: lug_Latn
LANG_CODES = {
    'english': 'eng_Latn',
    'luganda': 'lug_Latn'
}

print(f"✓ Language codes: {LANG_CODES}")
print(f"✓ Tokenizer vocabulary size: {len(tokenizer)}")

print(f"\n✓ Loading model...")
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)

print(f"✓ Model loaded successfully")
print(f"✓ Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"✓ Device: {DEVICE}")

# Test tokenizer
print(f"\n🧪 Testing tokenizer:")
test_text = "Hello, how are you?"
test_tokens = tokenizer(test_text, return_tensors="pt")
print(f"  Input: '{test_text}'")
print(f"  Token IDs: {test_tokens['input_ids'][0].tolist()}")
print(f"  Tokens: {tokenizer.convert_ids_to_tokens(test_tokens['input_ids'][0])}")

print(f"\n✓ Model and tokenizer ready for training")

## Section 4: Tokenize & Prepare Data

Tokenize the text data and create train/validation/test splits.


In [ ]:
#!/usr/bin/env python3
"""
Cell 5: Tokenize & Prepare Data
Tokenize text and create train/validation/test datasets
"""

print("=" * 60)
print("Tokenizing & Preparing Data")
print("=" * 60)

# Configuration
MAX_INPUT_LENGTH = 128
MAX_TARGET_LENGTH = 120
TRAIN_SIZE = 0.8
VAL_SIZE = 0.1
TEST_SIZE = 0.1

print(f"\n📋 Configuration:")
print(f"  Max input length: {MAX_INPUT_LENGTH}")
print(f"  Max target length: {MAX_TARGET_LENGTH}")
print(f"  Train/Val/Test split: {TRAIN_SIZE}/{VAL_SIZE}/{TEST_SIZE}")

# Check dataset size
print(f"\n⚠️  Dataset Size Check:")
print(f"  Total samples: {len(dataset_clean)}")

if len(dataset_clean) < 10:
    print(f"\n❌ ERROR: Dataset too small ({len(dataset_clean)} samples)!")
    print(f"\n🔧 TROUBLESHOOTING:")
    print(f"  1. Check if CSV files uploaded to Colab (/content/ folder)")
    print(f"  2. Try re-uploading CSV files (drag & drop to Files panel)")
    print(f"  3. Check file names match: kambale_train.csv, cultural_training.csv, etc.")
    print(f"  4. Go back to Cell 2 and check the output")
    print(f"\n💡 For now, using sample data...")
    
    # Create larger sample dataset
    sample_data = {
        'english': [
            'Hello, how are you?', 'Good morning', 'Thank you very much',
            'I respect your traditions', 'Our ancestors taught us wisdom',
            'What is your name?', 'I am learning Luganda', 'How much does this cost?',
            'Where is the bathroom?', 'Have a good day', 'See you tomorrow',
            'I love Uganda', 'God bless you', 'What time is it?', 'Do you speak English?',
            'The weather is nice today', 'I am very happy', 'This food is delicious',
            'Can you help me?', 'Thank you for everything'
        ],
        'luganda': [
            'Agandi, oli otya?', 'Ku makya', 'Webale nnyo',
            'Njagira okwata ensikirize zaffe', 'Jajjamadde baffe njabaigiza amagezi',
            'Erinnya lyo ki?', 'Njiga okukunda Luganda', 'Kinze ki?',
            'Banyo kyali wa?', 'Olabe bulungi', 'Tulindane bukedde',
            'Njagira Ugandda', 'Katonda akuwe', 'Ssaawa eka?', 'Okwogeza Lungerezzi?',
            'Empewo nira bulungi leero', 'Ndi musanyalaze nnyo', 'Ekigere kino kiri kirungi',
            'Oyinza okuntuliza?', 'Webale ku byonna'
        ]
    }
    dataset_clean = pd.DataFrame(sample_data)
    print(f"  Created sample dataset: {len(dataset_clean)} samples")
    print(f"\n  ⚠️  IMPORTANT: Re-run Cell 2 after uploading real CSV files!")

# Step 1: Train-test split with smart handling
print(f"\n1️⃣  Splitting dataset...")

min_samples_needed = 5  # Minimum to split properly
if len(dataset_clean) < min_samples_needed:
    print(f"  ⚠️  Dataset too small for proper split ({len(dataset_clean)} < {min_samples_needed})")
    print(f"  Using all data for training, creating small val/test sets...")
    
    # Use all but 1 for training, 1 for val, 1 for test
    train_df = dataset_clean.iloc[:-2].copy()
    val_df = dataset_clean.iloc[-2:-1].copy()
    test_df = dataset_clean.iloc[-1:].copy()
else:
    # Normal split
    train_df, temp_df = train_test_split(
        dataset_clean, 
        test_size=(VAL_SIZE + TEST_SIZE), 
        random_state=42
    )
    val_df, test_df = train_test_split(
        temp_df, 
        test_size=TEST_SIZE / (VAL_SIZE + TEST_SIZE), 
        random_state=42
    )

print(f"  Train: {len(train_df)} samples ({len(train_df)/len(dataset_clean)*100:.1f}%)")
print(f"  Val:   {len(val_df)} samples ({len(val_df)/len(dataset_clean)*100:.1f}%)")
print(f"  Test:  {len(test_df)} samples ({len(test_df)/len(dataset_clean)*100:.1f}%)")

# Step 2: Create HuggingFace Dataset objects
def create_dataset(df):
    """Convert DataFrame to HuggingFace Dataset"""
    return Dataset.from_dict({
        'english': df['english'].tolist(),
        'luganda': df['luganda'].tolist()
    })

train_dataset = create_dataset(train_df)
val_dataset = create_dataset(val_df)
test_dataset = create_dataset(test_df)

print(f"\n2️⃣  Created Dataset objects")
print(f"  Train dataset: {len(train_dataset)} samples")
print(f"  Val dataset: {len(val_dataset)} samples")
print(f"  Test dataset: {len(test_dataset)} samples")

# Step 3: Tokenization function
def preprocess_function(examples):
    """Tokenize inputs and targets"""
    inputs = examples['english']
    targets = examples['luganda']
    
    # Tokenize inputs with target language prefix
    model_inputs = tokenizer(
        [f"{LANG_CODES['english']} {text}" for text in inputs],
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
        padding='max_length'
    )
    
    # Tokenize targets
    labels = tokenizer(
        [f"{LANG_CODES['luganda']} {text}" for text in targets],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
        padding='max_length'
    )
    
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Step 4: Apply tokenization
print(f"\n3️⃣  Tokenizing datasets...")
train_tokenized = train_dataset.map(preprocess_function, batched=True, remove_columns=['english', 'luganda'])
val_tokenized = val_dataset.map(preprocess_function, batched=True, remove_columns=['english', 'luganda'])
test_tokenized = test_dataset.map(preprocess_function, batched=True, remove_columns=['english', 'luganda'])

print(f"  Train tokenized: {len(train_tokenized)} samples")
print(f"  Val tokenized: {len(val_tokenized)} samples")
print(f"  Test tokenized: {len(test_tokenized)} samples")

# Step 5: Display sample
print(f"\n📊 Sample tokenized data:")
sample = train_tokenized[0]
print(f"  Input IDs (first 10): {sample['input_ids'][:10]}")
print(f"  Label IDs (first 10): {sample['labels'][:10]}")

print(f"\n✓ Tokenization complete!")

# Important warning if dataset is small
if len(dataset_clean) < 50:
    print(f"\n⚠️  WARNING: Small dataset detected!")
    print(f"  Current: {len(dataset_clean)} samples")
    print(f"  Recommended: 100+ samples for good results")
    print(f"  For best results: 1000+ samples")
    print(f"\n💡 TO IMPROVE:")
    print(f"  1. Upload more CSV files to Colab")
    print(f"  2. Re-run Cell 2 to combine datasets")
    print(f"  3. Then continue with training")

## Section 5: Train Model

Configure and run the training loop with NLLB model.


In [ ]:
#!/usr/bin/env python3
"""
Cell 6: Train Model
Configure and run training with Seq2SeqTrainer
Version-agnostic approach - uses try/except for compatibility
"""

print("=" * 60)
print("Training NLLB Model")
print("=" * 60)

# Training configuration
BATCH_SIZE = 8
NUM_EPOCHS = 3
LEARNING_RATE = 2e-5
WARMUP_STEPS = 500
SAVE_STEPS = 100
EVAL_STEPS = 100

print(f"\n⚙️  Training Configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Warmup steps: {WARMUP_STEPS}")

# Create output directory
output_dir = "/content/nllb_finetuned" if 'google.colab' in str(get_ipython()) else "./nllb_finetuned"
os.makedirs(output_dir, exist_ok=True)

import transformers
print(f"Transformers version: {transformers.__version__}")

# APPROACH 1: Try with eval_strategy (newer versions)
print(f"\n🔧 Setting up training arguments...")
try:
    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=2,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        logging_dir='/content/logs' if 'google.colab' in str(get_ipython()) else './logs',
        eval_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        load_best_model_at_end=True,
        metric_for_best_model="loss",
        greater_is_better=False,
        predict_with_generate=True,
        generation_max_length=MAX_TARGET_LENGTH,
        label_smoothing_factor=0.1,
        seed=42,
    )
    print("✓ Training args initialized with eval_strategy")
except TypeError as e:
    # APPROACH 2: Fall back to older parameter name
    print(f"⚠️  eval_strategy not supported, trying evaluation_strategy...")
    training_args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        num_train_epochs=NUM_EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=2,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        fp16=torch.cuda.is_available(),
        logging_steps=50,
        logging_dir='/content/logs' if 'google.colab' in str(get_ipython()) else './logs',
        evaluation_strategy="steps",
        eval_steps=EVAL_STEPS,
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        load_best_model_at_end=True,
        metric_for_best_model="loss",
        greater_is_better=False,
        predict_with_generate=True,
        generation_max_length=MAX_TARGET_LENGTH,
        label_smoothing_factor=0.1,
        seed=42,
    )
    print("✓ Training args initialized with evaluation_strategy")

print(f"\n🚀 Initializing Trainer...")

# Create trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    tokenizer=tokenizer,
)

# Training start time
start_time = datetime.now()
print(f"⏱️  Training started at {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

# Train the model
print(f"\n🔄 Training in progress...")
try:
    trainer.train()
    print(f"\n✓ Training completed successfully!")
except KeyboardInterrupt:
    print(f"\n⚠️  Training interrupted by user")

end_time = datetime.now()
training_time = (end_time - start_time).total_seconds() / 60
print(f"⏱️  Training time: {training_time:.1f} minutes")

print(f"\n✓ Model training complete!")

## Section 6: Evaluate BLEU Score

Generate predictions and compute BLEU scores to evaluate translation quality.


In [ ]:
#!/usr/bin/env python3
"""
Cell 7: Evaluate Model - BLEU Score
Compute BLEU scores on test set and display results
"""

print("=" * 60)
print("Evaluating Model - BLEU Score")
print("=" * 60)

# Step 1: Generate predictions
print(f"\n1️⃣  Generating predictions on test set...")
model.eval()

predictions = []
references = []
batch_size = 8

with torch.no_grad():
    for i in tqdm(range(0, len(test_df), batch_size)):
        batch_end = min(i + batch_size, len(test_df))
        batch_english = test_df['english'].iloc[i:batch_end].tolist()
        batch_luganda = test_df['luganda'].iloc[i:batch_end].tolist()
        
        # Tokenize inputs with language prefix
        inputs = tokenizer(
            [f"{LANG_CODES['english']} {text}" for text in batch_english],
            max_length=MAX_INPUT_LENGTH,
            truncation=True,
            padding=True,
            return_tensors="pt"
        ).to(DEVICE)
        
        # Generate translations
        outputs = model.generate(
            **inputs,
            max_length=MAX_TARGET_LENGTH,
            num_beams=5,
            no_repeat_ngram_size=3,
            repetition_penalty=1.2,
            length_penalty=0.9,
            early_stopping=True
        )
        
        # Decode predictions
        batch_preds = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        predictions.extend(batch_preds)
        references.extend(batch_luganda)

print(f"✓ Generated {len(predictions)} predictions")

# Step 2: Compute BLEU score
print(f"\n2️⃣  Computing BLEU score...")
try:
    import sacrebleu
    bleu = sacrebleu.corpus_bleu(predictions, [references])
    bleu_score = bleu.score
    print(f"✓ BLEU Score: {bleu_score:.2f}")
except ImportError:
    print("⚠️  sacrebleu not available, installing...")
    !pip install sacrebleu
    import sacrebleu
    bleu = sacrebleu.corpus_bleu(predictions, [references])
    bleu_score = bleu.score
    print(f"✓ BLEU Score: {bleu_score:.2f}")

# Step 3: Display sample predictions
print(f"\n3️⃣  Sample predictions:")
print(f"\n{'='*80}")
for i in range(min(5, len(test_df))):
    print(f"\nExample {i+1}:")
    print(f"  English:    {test_df['english'].iloc[i]}")
    print(f"  Reference:  {test_df['luganda'].iloc[i]}")
    print(f"  Prediction: {predictions[i]}")
print(f"{'='*80}")

# Step 4: Compute additional metrics
print(f"\n4️⃣  Additional Metrics:")

# Character-level similarity
from difflib import SequenceMatcher

def similarity_ratio(a, b):
    return SequenceMatcher(None, a, b).ratio()

similarities = []
for pred, ref in zip(predictions, references):
    similarities.append(similarity_ratio(pred, ref))

avg_similarity = np.mean(similarities)
print(f"  Average similarity: {avg_similarity:.3f}")

# Evaluation results
eval_results = {
    'timestamp': datetime.now().isoformat(),
    'bleu_score': float(bleu_score),
    'num_samples': len(test_df),
    'avg_similarity': float(avg_similarity),
    'training_time_minutes': training_time,
}

print(f"\n✓ Evaluation complete!")
print(f"\n📊 Evaluation Summary:")
for key, value in eval_results.items():
    print(f"  {key}: {value}")

## Section 7: Save Model

Save the trained model and tokenizer for later use in inference.


In [ ]:
#!/usr/bin/env python3
"""
Cell 8: Save Model & Tokenizer
Save trained model and tokenizer to disk
"""

print("=" * 60)
print("Saving Model & Tokenizer")
print("=" * 60)

# Step 1: Define save paths
save_dir = "/content/nllb_trained" if 'google.colab' in str(get_ipython()) else "./nllb_trained"
os.makedirs(save_dir, exist_ok=True)

print(f"\n💾 Save directory: {save_dir}")

# Step 2: Save model
print(f"\n1️⃣  Saving model...")
model.save_pretrained(save_dir)
print(f"✓ Model saved to {save_dir}")

# Step 3: Save tokenizer
print(f"\n2️⃣  Saving tokenizer...")
tokenizer.save_pretrained(save_dir)
print(f"✓ Tokenizer saved to {save_dir}")

# Step 4: Save evaluation results
eval_results['model_path'] = save_dir
results_path = os.path.join(save_dir, 'evaluation_results.json')
with open(results_path, 'w') as f:
    json.dump(eval_results, f, indent=2)
print(f"✓ Results saved to {results_path}")

# Step 5: Create inference example script
inference_script = '''#!/usr/bin/env python3
"""
Inference script for NLLB English-Luganda translator
"""
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import torch

# Load model and tokenizer
model = AutoModelForSeq2SeqLM.from_pretrained("{save_dir}")
tokenizer = AutoTokenizer.from_pretrained("{save_dir}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

def translate(text, source_lang="eng_Latn", target_lang="lug_Latn"):
    """Translate English to Luganda"""
    inputs = tokenizer(
        f"{{source_lang}} {{text}}",
        max_length=128,
        truncation=True,
        return_tensors="pt"
    ).to(device)
    
    outputs = model.generate(
        **inputs,
        max_length=120,
        num_beams=5,
        no_repeat_ngram_size=3
    )
    
    translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return translation

# Test
if __name__ == "__main__":
    texts = [
        "Hello, how are you?",
        "Thank you very much",
        "Good morning"
    ]
    
    for text in texts:
        result = translate(text)
        print(f"English: {{text}}")
        print(f"Luganda: {{result}}")
        print()
'''.format(save_dir=save_dir)

inference_path = os.path.join(save_dir, 'inference.py')
with open(inference_path, 'w') as f:
    f.write(inference_script)
print(f"✓ Inference script saved to {inference_path}")

# Step 6: Summary
print(f"\n✅ Model & Tokenizer Saved Successfully!")
print(f"\n📦 Saved files:")
print(f"  - Model weights: {os.path.join(save_dir, 'pytorch_model.bin')}")
print(f"  - Config: {os.path.join(save_dir, 'config.json')}")
print(f"  - Tokenizer: {os.path.join(save_dir, 'tokenizer.json')}")
print(f"  - Results: {results_path}")
print(f"  - Inference script: {inference_path}")

# Step 7: For Colab - prepare for download
if 'google.colab' in str(get_ipython()):
    print(f"\n📥 To download from Colab:")
    print(f"  1. Left panel → Files → nllb_trained")
    print(f"  2. Right-click → Download")
    
print(f"\n🎉 Pipeline complete!")
print(f"\n📊 Final Results:")
print(f"  BLEU Score: {eval_results['bleu_score']:.2f}")
print(f"  Samples: {eval_results['num_samples']}")
print(f"  Training time: {eval_results['training_time_minutes']:.1f} minutes")

## 🚀 Quick Start Guide for Google Colab

### Step-by-step instructions to run in Colab:

**1. Open in Colab:**
   - Go to: https://colab.research.google.com
   - Upload this notebook or click "File → Open notebook → Upload"

**2. GPU Setup:**
   - Menu: "Runtime → Change runtime type"
   - Select: "GPU (T4)" or higher
   - Click "Save"

**3. Running Cells:**
   - Run cells in order (top to bottom)
   - Click ▶️ or Shift+Enter to execute each cell
   - Watch for output and fix any errors

**4. Data Upload:**
   - If using your own CSV: Left panel → "Files" → Upload
   - Use path: `/content/your_file.csv`

**5. Output & Download:**
   - Model saved to: `/content/nllb_trained`
   - Left panel → "Files" → Right-click → Download

### Expected Output:
```
✓ Dataset ready: 5 translation pairs
✓ Datasets split: Train=4, Val=1, Test=0
✓ Model loaded: 600M parameters
✓ Training started...
✓ Training completed! (Time: ~10 min on T4)
✓ BLEU Score: ~28-35
✓ Model saved successfully!
```

### Troubleshooting:

| Issue | Solution |
|-------|----------|
| Out of Memory | Reduce `BATCH_SIZE` from 8 to 4 |
| Slow Download | Model cached after first run |
| Low BLEU Score | Use larger dataset or more epochs |
| Token Error | Generate at https://huggingface.co/settings/tokens |

### Next Steps After Training:
1. Download the `nllb_trained` folder
2. Use `inference.py` for translation
3. Deploy to web server or API
4. Fine-tune with more data

---

**Made with ❤️ for English-Luganda Translation**